# Recommender Systems: Collaborative Filtering, Factorization & Ranking Lab

Recommender systems match users with personalized items within massive catalogs. This lab implements Matrix Factorization (FunkSVD) with latent user/item embeddings, explores content-based cold-start mitigation, and computes production ranking metrics (Precision@K, NDCG@K, MRR).

### Table of Contents
1. [User-Item Rating Matrices & The Sparsity Challenge](#1-user-item-rating-matrices--the-sparsity-challenge)
2. [Matrix Factorization via FunkSVD (Latent Embedding Learning)](#2-matrix-factorization-via-funksvd-latent-embedding-learning)
3. [Ranking Metrics: Precision@K, Recall@K & NDCG@K](#3-ranking-metrics-precisionk-recallk--ndcgk)
4. [Cold-Start Problem & Content-Based Hybrid Solutions](#4-cold-start-problem--content-based-hybrid-solutions)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
print("Loaded Recommender Systems interactive environment.")

## 1. User-Item Rating Matrices & The Sparsity Challenge
In industrial platforms (Netflix, Amazon), users rate fewer than $0.1\%$ of available items. Visualizing the sparse user-item interaction matrix.

In [ ]:
# Simulate sparse rating matrix: 20 users x 15 movies (90% unobserved)
np.random.seed(42)
n_users, n_items = 20, 15
true_latent = np.random.uniform(1, 5, size=(n_users, n_items))
mask = np.random.binomial(1, 0.15, size=(n_users, n_items)).astype(bool)
R_sparse = np.where(mask, true_latent, np.nan)

plt.figure(figsize=(8, 4.5))
sns.heatmap(~np.isnan(R_sparse), cmap="Blues", cbar=False, yticklabels=False, xticklabels=False)
plt.title(f"Observed Ratings Matrix (Sparsity: {1.0 - np.mean(mask):.1%})", fontweight="bold")
plt.xlabel("Items (Movies)")
plt.ylabel("Users")
plt.tight_layout()
plt.show()

## 2. Matrix Factorization via FunkSVD
Decomposing $R \approx \mu + b_u + b_i + P Q^T$ using Stochastic Gradient Descent on observed ratings.

In [ ]:
n_factors = 4
lr = 0.02
reg = 0.05
epochs = 100

mu = np.nanmean(R_sparse)
b_u = np.zeros(n_users)
b_i = np.zeros(n_items)
P = np.random.normal(0, 0.1, size=(n_users, n_factors))
Q = np.random.normal(0, 0.1, size=(n_items, n_factors))

users, items = np.where(~np.isnan(R_sparse))
loss_history = []

for _ in range(epochs):
    total_loss = 0.0
    for u, i in zip(users, items):
        pred = mu + b_u[u] + b_i[i] + np.dot(P[u], Q[i])
        err = R_sparse[u, i] - pred
        total_loss += err ** 2
        
        b_u[u] += lr * (err - reg * b_u[u])
        b_i[i] += lr * (err - reg * b_i[i])
        p_old = P[u].copy()
        P[u] += lr * (err * Q[i] - reg * P[u])
        Q[i] += lr * (err * p_old - reg * Q[i])
    loss_history.append(total_loss / len(users))

plt.figure(figsize=(7, 4))
plt.plot(loss_history, color="#1f77b4", lw=2)
plt.title("FunkSVD Reconstruction Training Loss", fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("MSE on Observed Ratings")
plt.tight_layout()
plt.show()

## 3. Ranking Evaluation Metrics (NDCG@K, Precision@K)
Evaluating the quality of top-$K$ recommendations using Discounted Cumulative Gain.

In [ ]:
# Simulated user recommendations
ground_truth_liked = {3, 7, 12, 14}
recommended_rankings = [7, 2, 14, 5, 3, 9, 1]

def evaluate_rankings(actual, recs, k=5):
    top_k = recs[:k]
    hits = len(set(top_k) & actual)
    prec = hits / k
    rec = hits / len(actual)
    
    dcg = sum(1.0 / np.log2(i + 1) for i, item in enumerate(top_k, 1) if item in actual)
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, min(len(actual), k) + 1))
    ndcg = dcg / idcg if idcg > 0 else 0.0
    return prec, rec, ndcg

p5, r5, ndcg5 = evaluate_rankings(ground_truth_liked, recommended_rankings, k=5)
print(f"Precision@5: {p5:.2%}")
print(f"Recall@5:    {r5:.2%}")
print(f"NDCG@5:      {ndcg5:.4f}")